## Nomor 1 — Membaca Dataset Titanic
Membaca file titanic.csv dan menampilkan isi serta ukuran dataset.

In [1]:
import pandas as pd

dataset = pd.read_csv('titanic.csv')

print("Dataset Titanic : ")
print(dataset)

print("\nJumlah baris;", dataset.shape[0])
print("Jumlah kolom;", dataset.shape[1])

Dataset Titanic : 
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                

## Nomor 2 — Menyiapkan Metode Validasi
Membagi indeks dataset menggunakan Hold-out 70%–30%, K-Fold 10,
dan Leave-One-Out (LOO).

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split, KFold, LeaveOneOut

indeks = np.arange(len(dataset))

# a. Hold-out: 70% training dan 30% test
train_idx, test_idx = train_test_split(
    indeks,
    test_size=0.30,
    random_state=42
)

holdout_split = [(train_idx, test_idx)]

print("a. Hold-out 70%–30%")
print("Jumlah data training:", len(train_idx))
print("Jumlah data test:", len(test_idx))


# b. K-Fold: 10 bagian
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
kfold_split = list(kfold.split(dataset))

print("\nb. K-Fold (10 fold)")

for nomor, (train_idx, test_idx) in enumerate(kfold_split, start=1):
    print(
        f"Fold {nomor}: "
        f"{len(train_idx)} data training, "
        f"{len(test_idx)} data test"
    )


# c. Leave-One-Out: setiap data menjadi test satu kali
loo = LeaveOneOut()
loo_split = list(loo.split(dataset))

print("\nc. Leave-One-Out (LOO)")
print("Jumlah percobaan:", len(loo_split))
print("Jumlah data training per percobaan:", len(loo_split[0][0]))
print("Jumlah data test per percobaan:", len(loo_split[0][1]))


# Simpan semua pembagian untuk digunakan pada langkah berikutnya
metode_validasi = {
    "Hold-out": holdout_split,
    "K-Fold": kfold_split,
    "LOO": loo_split
}

a. Hold-out 70%–30%
Jumlah data training: 623
Jumlah data test: 268

b. K-Fold (10 fold)
Fold 1: 801 data training, 90 data test
Fold 2: 802 data training, 89 data test
Fold 3: 802 data training, 89 data test
Fold 4: 802 data training, 89 data test
Fold 5: 802 data training, 89 data test
Fold 6: 802 data training, 89 data test
Fold 7: 802 data training, 89 data test
Fold 8: 802 data training, 89 data test
Fold 9: 802 data training, 89 data test
Fold 10: 802 data training, 89 data test

c. Leave-One-Out (LOO)
Jumlah percobaan: 891
Jumlah data training per percobaan: 890
Jumlah data test per percobaan: 1


## Nomor 3 — Menyiapkan Fitur dan Mengisi Missing Values
Mengambil fitur Sex, Age, Pclass, dan Fare, mengubah Sex menjadi angka,
serta mengisi Age yang kosong dengan rata-rata usia berdasarkan Pclass
dari data training pada setiap pembagian validasi.

In [3]:
fitur = ["Sex", "Age", "Pclass", "Fare"]
train_data = dataset[fitur].copy()

train_data["Sex"] = train_data["Sex"].map({
    "male": 0,
    "female": 1
})


def siapkan_data(train_idx, test_idx):
    data_train = train_data.iloc[train_idx].copy()
    data_test = train_data.iloc[test_idx].copy()

    kelas_train = dataset["Survived"].iloc[train_idx]
    kelas_test = dataset["Survived"].iloc[test_idx]

    # Mean dihitung dari training, dikelompokkan menurut Survived
    mean_age = (
        dataset.iloc[train_idx]
        .groupby("Survived")["Age"]
        .mean()
    )

    pengisi_train = kelas_train.map(mean_age)
    pengisi_test = kelas_test.map(mean_age)

    data_train["Age"] = data_train["Age"].fillna(pengisi_train)

    # Mengikuti contoh praktikum:
    # label asli test digunakan untuk menentukan pengisian Age.
    # Penggunaan ini menyebabkan kebocoran label dalam evaluasi.
    data_test["Age"] = data_test["Age"].fillna(pengisi_test)

    return data_train, data_test, mean_age


train_idx, test_idx = holdout_split[0]
data_train, data_test, mean_age = siapkan_data(train_idx, test_idx)

print("Mean Age berdasarkan kelas Survived pada training:")
print(mean_age)

print("\nData training setelah pengisian Age:")
print(data_train)

print("\nData test setelah pengisian Age:")
print(data_test)

print("\nMissing values training:")
print(data_train.isna().sum())

print("\nMissing values test:")
print(data_test.isna().sum())

Mean Age berdasarkan kelas Survived pada training:
Survived
0    30.186667
1    27.853869
Name: Age, dtype: float64

Data training setelah pengisian Age:
     Sex        Age  Pclass      Fare
445    0   4.000000       1   81.8583
650    0  30.186667       3    7.8958
172    1   1.000000       3   11.1333
450    0  36.000000       2   27.7500
314    0  43.000000       2   26.2500
..   ...        ...     ...       ...
106    1  21.000000       3    7.6500
270    0  30.186667       1   31.0000
860    0  41.000000       3   14.1083
435    1  14.000000       1  120.0000
102    0  21.000000       1   77.2875

[623 rows x 4 columns]

Data test setelah pengisian Age:
     Sex        Age  Pclass     Fare
709    0  27.853869       3  15.2458
439    0  31.000000       2  10.5000
840    0  20.000000       3   7.9250
720    1   6.000000       2  33.0000
39     1  14.000000       3  11.2417
..   ...        ...     ...      ...
821    0  27.000000       3   8.6625
633    0  30.186667       1   0.0000

## Nomor 4 — Mengambil Label
Mengambil kolom Survived sebagai label klasifikasi:
0 berarti tidak selamat dan 1 berarti selamat.

In [4]:
label = dataset["Survived"].copy()

print("Label dataset:")
print(label)

print("\nJumlah label:", len(label))

print("\nJumlah data setiap kelas:")
print(label.value_counts().sort_index())

# Contoh pembagian label untuk Hold-out
train_idx, test_idx = holdout_split[0]

train_label = label.iloc[train_idx]
test_label = label.iloc[test_idx]

print("\nLabel training Hold-out:")
print(train_label)

print("\nLabel test Hold-out:")
print(test_label)

print("\nJumlah label training:", len(train_label))
print("Jumlah label test:", len(test_label))

Label dataset:
0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

Jumlah label: 891

Jumlah data setiap kelas:
Survived
0    549
1    342
Name: count, dtype: int64

Label training Hold-out:
445    1
650    0
172    1
450    0
314    0
      ..
106    1
270    0
860    0
435    1
102    0
Name: Survived, Length: 623, dtype: int64

Label test Hold-out:
709    1
439    0
840    0
720    1
39     1
      ..
821    1
633    0
456    0
500    0
430    1
Name: Survived, Length: 268, dtype: int64

Jumlah label training: 623
Jumlah label test: 268


## Nomor 5 — Normalisasi Data Training
Melakukan normalisasi Min-Max 0–1 pada data training serta mencatat
nilai minimum dan maksimum setiap fitur.

Rumus: x_normalisasi = (x - minimum) / (maksimum - minimum)

In [5]:
# Fungsi ini digunakan kembali pada setiap pembagian validasi
def normalisasi_training(data_train):
    min_train = data_train.min()
    max_train = data_train.max()

    # Hindari pembagian nol jika suatu fitur memiliki nilai konstan
    rentang_train = (max_train - min_train).replace(0, 1)

    train_normal = (data_train - min_train) / rentang_train

    return train_normal, min_train, max_train, rentang_train


# Normalisasi training Hold-out dari nomor 3
train_normal, min_train, max_train, rentang_train = (
    normalisasi_training(data_train)
)

print("Minimum dan maksimum setiap fitur training:")
print(pd.DataFrame({
    "Minimum": min_train,
    "Maksimum": max_train
}))

print("\nData training setelah normalisasi:")
print(train_normal)

print("\nRentang nilai setelah normalisasi:")
print(pd.DataFrame({
    "Minimum": train_normal.min(),
    "Maksimum": train_normal.max()
}))

Minimum dan maksimum setiap fitur training:
        Minimum  Maksimum
Sex        0.00    1.0000
Age        0.42   80.0000
Pclass     1.00    3.0000
Fare       0.00  512.3292

Data training setelah normalisasi:
     Sex       Age  Pclass      Fare
445  0.0  0.044986     0.0  0.159777
650  0.0  0.374047     1.0  0.015412
172  1.0  0.007288     1.0  0.021731
450  0.0  0.447097     0.5  0.054164
314  0.0  0.535059     0.5  0.051237
..   ...       ...     ...       ...
106  1.0  0.258608     1.0  0.014932
270  0.0  0.374047     0.0  0.060508
860  0.0  0.509927     1.0  0.027538
435  1.0  0.170646     0.0  0.234224
102  0.0  0.258608     0.0  0.150855

[623 rows x 4 columns]

Rentang nilai setelah normalisasi:
        Minimum  Maksimum
Sex         0.0       1.0
Age         0.0       1.0
Pclass      0.0       1.0
Fare        0.0       1.0


## Nomor 6 — Normalisasi Data Test
Melakukan normalisasi data test menggunakan nilai minimum dan maksimum
dari data training pada nomor 5.

In [6]:
# Fungsi ini digunakan kembali pada setiap pembagian validasi
def normalisasi_test(data_test, min_train, rentang_train):
    return (data_test - min_train) / rentang_train


# Normalisasi test Hold-out dengan parameter training dari nomor 5
test_normal = normalisasi_test(
    data_test, min_train, rentang_train
)

print("Minimum dan maksimum training yang digunakan:")
print(pd.DataFrame({
    "Minimum": min_train,
    "Maksimum": max_train
}))

print("\nData test sebelum normalisasi:")
print(data_test)

print("\nData test setelah normalisasi:")
print(test_normal)

print("\nJumlah data test:", len(test_normal))
print("Jumlah missing values setelah normalisasi:")
print(test_normal.isna().sum())

Minimum dan maksimum training yang digunakan:
        Minimum  Maksimum
Sex        0.00    1.0000
Age        0.42   80.0000
Pclass     1.00    3.0000
Fare       0.00  512.3292

Data test sebelum normalisasi:
     Sex        Age  Pclass     Fare
709    0  27.853869       3  15.2458
439    0  31.000000       2  10.5000
840    0  20.000000       3   7.9250
720    1   6.000000       2  33.0000
39     1  14.000000       3  11.2417
..   ...        ...     ...      ...
821    0  27.000000       3   8.6625
633    0  30.186667       1   0.0000
456    0  65.000000       1  26.5500
500    0  17.000000       3   8.6625
430    0  28.000000       1  26.5500

[268 rows x 4 columns]

Data test setelah normalisasi:
     Sex       Age  Pclass      Fare
709  0.0  0.344733     1.0  0.029758
439  0.0  0.384267     0.5  0.020495
840  0.0  0.246042     1.0  0.015469
720  1.0  0.070118     0.5  0.064412
39   1.0  0.170646     1.0  0.021942
..   ...       ...     ...       ...
821  0.0  0.334004     1.0  0.016

## Nomor 7 — Klasifikasi k-NN dan Evaluasi
Melakukan klasifikasi k-NN dengan k = 3 menggunakan metode Hold-out
70%–30%, K-Fold 10, dan Leave-One-Out (LOO).

Error ratio = jumlah prediksi salah / jumlah seluruh prediksi test.

In [7]:
from sklearn.neighbors import KNeighborsClassifier

ringkasan = []

for nama_metode, daftar_split in metode_validasi.items():
    catatan = []

    for nomor, (train_idx, test_idx) in enumerate(
        daftar_split, start=1
    ):
        data_train, data_test, _ = siapkan_data(train_idx, test_idx)

        y_train = dataset["Survived"].iloc[train_idx]
        y_test = dataset["Survived"].iloc[test_idx]

        train_normal, nilai_min, nilai_max, rentang = (
            normalisasi_training(data_train)
        )
        test_normal = normalisasi_test(
            data_test, nilai_min, rentang
        )

        model = KNeighborsClassifier(n_neighbors=3)
        model.fit(train_normal, y_train)
        prediksi = model.predict(test_normal)

        salah = int((prediksi != y_test.to_numpy()).sum())
        jumlah = len(y_test)

        catatan.append({
            "Percobaan": nomor,
            "Jumlah Test": jumlah,
            "Benar": jumlah - salah,
            "Salah": salah,
            "Error Ratio": salah / jumlah
        })

    detail = pd.DataFrame(catatan)

    total_test = int(detail["Jumlah Test"].sum())
    total_salah = int(detail["Salah"].sum())
    total_benar = total_test - total_salah

    # Rata-rata error percobaan, seperti pada laporan teman
    error_ratio = detail["Error Ratio"].mean()
    akurasi = 1 - error_ratio

    print(f"\n=== {nama_metode} ===")

    if nama_metode != "LOO":
        print(detail.to_string(index=False))

    print("Jumlah percobaan:", len(detail))
    print("Total prediksi test:", total_test)
    print("Prediksi benar:", total_benar)
    print("Prediksi salah:", total_salah)
    print(f"Rata-rata akurasi: {akurasi * 100:.2f}%")
    print(f"Rata-rata error ratio: {error_ratio:.6f}")
    print(f"Rata-rata error (%): {error_ratio * 100:.2f}%")

    ringkasan.append({
        "Metode": nama_metode,
        "Benar": total_benar,
        "Salah": total_salah,
        "Akurasi (%)": akurasi * 100,
        "Error (%)": error_ratio * 100
    })

hasil_evaluasi = pd.DataFrame(ringkasan)

print("\n=== Ringkasan Evaluasi ===")
print(hasil_evaluasi.to_string(index=False, float_format="%.2f"))


=== Hold-out ===
 Percobaan  Jumlah Test  Benar  Salah  Error Ratio
         1          268    228     40     0.149254
Jumlah percobaan: 1
Total prediksi test: 268
Prediksi benar: 228
Prediksi salah: 40
Rata-rata akurasi: 85.07%
Rata-rata error ratio: 0.149254
Rata-rata error (%): 14.93%

=== K-Fold ===
 Percobaan  Jumlah Test  Benar  Salah  Error Ratio
         1           90     77     13     0.144444
         2           89     74     15     0.168539
         3           89     78     11     0.123596
         4           89     74     15     0.168539
         5           89     79     10     0.112360
         6           89     78     11     0.123596
         7           89     71     18     0.202247
         8           89     75     14     0.157303
         9           89     69     20     0.224719
        10           89     77     12     0.134831
Jumlah percobaan: 10
Total prediksi test: 891
Prediksi benar: 752
Prediksi salah: 139
Rata-rata akurasi: 84.40%
Rata-rata error ratio